In [1]:
"""
Clinical Trial Criteria Analysis Script
========================================
Analyzes pipeline output to generate descriptive statistics and category distributions.

Produces two tables:
1. Descriptive Statistics: Criterion count, average words, attributes, questions, and options
2. Category Distribution: Percentage breakdown of criteria by primary category
"""

import os
import pandas as pd
import numpy as np
import re
from typing import Dict, List, Tuple


# ================================================================
# CONFIGURATION
# ================================================================

ROOT_DIR = "../"
OUTPUT_DIR_QUES = "output/question/"
RESULTS_DIR = "output/analysis/"

# Diseases to analyze
DISEASES = ['Alzheimer', 'Diabetes', 'NSCLC', 'UC', 'AML', 'CF', 'MS']

# Model to analyze (change as needed)
MODEL = 'gpt-5.1'

# Category mappings for Table 2
# Maps secondary categories to primary categories for reporting
CATEGORY_GROUPS = {
    'Comorbidity': [
        'Cardiovascular', 'Kidney', 'Neurological', 'Endocrine', 
        'Gastrointestinal', 'Infectious_Disease', 'Malignancy', 
        'Autoimmune', 'Other_Complication'
    ],
    'Diagnosis': ['Diagnosis', 'Severity', 'Biomarkers'],
    'Laboratory': ['Laboratory', 'Vital_Signs', 'Body_Measures'],
    'Neurological': ['Cognitive_Function'],  # If tracked separately
    'Participation': [
        'Language', 'Geographic', 'Logistics', 'General_Health', 
        'Others', 'Pregnancy', 'Contraception', 'Lactation',
        'Physical_Activity', 'Substance_Use', 'Nutrition'
    ],
    'Treatment': [
        'Current_Treatment', 'Treatment_History', 'Medication_Exclusion',
        'Immunosuppression', 'Device_Use', 'Transplant', 'Vaccination',
        'Adverse_Events'
    ]
}


# ================================================================
# HELPER FUNCTIONS
# ================================================================

def ensure_dir(path: str) -> None:
    """Create directory if it doesn't exist."""
    os.makedirs(path, exist_ok=True)


def count_words(text: str) -> int:
    """Count words in a text string."""
    if pd.isna(text) or text == '':
        return 0
    return len(str(text).split())


def is_excluded_option(option_text: str) -> bool:
    """
    Check if option should be excluded from counting.
    Excludes: 'None of the above', options starting with 'No', 
    and negative options in binary yes/no questions.
    """
    if pd.isna(option_text):
        return True
    
    option_lower = str(option_text).lower().strip()
    
    # Exclude "None of the above"
    if 'none of the above' in option_lower:
        return True
    
    # Exclude "Don't know" or "Unknown"
    if option_lower in ["don't know", "unknown", "unsure", "not sure"]:
        return True
    
    # Exclude options starting with "No"
    if option_lower.startswith('no ') or option_lower == 'no':
        return True
    
    return False


def map_to_category_group(secondary_category: str) -> str:
    """Map secondary category to primary category group."""
    if pd.isna(secondary_category):
        return 'Other'
    
    secondary_category = str(secondary_category).strip()
    
    for group, categories in CATEGORY_GROUPS.items():
        if secondary_category in categories:
            return group
    
    return 'Other'


# ================================================================
# ANALYSIS FUNCTIONS
# ================================================================

def analyze_disease(disease: str, model: str) -> Dict[str, any]:
    """
    Analyze a single disease and return statistics.
    
    Returns dictionary with:
    - criterion_count: Total number of criteria
    - avg_words: Average words per criterion
    - avg_attributes: Average attributes per criterion
    - avg_questions: Average questions per criterion
    - avg_options: Average valid options per criterion
    - category_dist: Distribution of criteria by category
    """
    
    # Load the question generation output file
    input_file = os.path.join(
        ROOT_DIR, OUTPUT_DIR_QUES, f"ques_{model}_{disease}.xlsx"
    )
    
    if not os.path.exists(input_file):
        print(f"Warning: File not found for {disease}: {input_file}")
        return None
    
    # Load all sheets
    df_option_attr = pd.read_excel(input_file, sheet_name='option_attribute')
    df_attribute = pd.read_excel(input_file, sheet_name='attribute')
    df_subcriterion = pd.read_excel(input_file, sheet_name='subcriterion')
    df_question = pd.read_excel(input_file, sheet_name='question')
    
    # Get criterion-level data by merging with attributes
    # Use the 'criterion' sheet if available, otherwise derive from subcriteria
    try:
        df_criterion = pd.read_excel(input_file, sheet_name='criterion')
    except:
        # Derive from subcriteria
        df_criterion = df_subcriterion.drop_duplicates('Criterion ID')[
            ['Criterion ID', 'Criterion', 'Type']
        ].copy()
    
    # 1. Criterion Count
    criterion_count = len(df_criterion)
    
    # 2. Average Words per Criterion
    df_criterion['word_count'] = df_criterion['Criterion'].apply(count_words)
    avg_words = df_criterion['word_count'].mean()
    
    # 3. Average Attributes per Criterion
    attr_per_criterion = df_attribute.groupby('Criterion ID').size()
    avg_attributes = attr_per_criterion.mean()
    
    # 4. Average Questions per Criterion
    # Map questions to criteria through attributes
    df_option_attr_valid = df_option_attr[df_option_attr['Criterion ID'].notna()].copy()
    questions_per_criterion = df_option_attr_valid.groupby('Criterion ID')['Question ID'].nunique()
    avg_questions = questions_per_criterion.mean()
    
    # 5. Average Valid Options per Criterion
    # First, identify valid options (excluding "None of the above", etc.)
    df_option_attr_valid['is_excluded'] = df_option_attr_valid['Option Text'].apply(
        is_excluded_option
    )
    df_valid_options = df_option_attr_valid[~df_option_attr_valid['is_excluded']]
    
    # Count unique valid options per criterion
    options_per_criterion = df_valid_options.groupby('Criterion ID').apply(
        lambda x: x[['Question ID', 'Option Text']].drop_duplicates().shape[0]
    )
    avg_options = options_per_criterion.mean()
    
    # 6. Category Distribution
    # Map secondary categories to primary groups
    df_attribute['Category_Group'] = df_attribute['Secondary Category'].apply(
        map_to_category_group
    )
    
    # Get one row per criterion with its category
    df_criterion_category = df_attribute.drop_duplicates('Criterion ID')[
        ['Criterion ID', 'Category_Group']
    ]
    
    # Calculate distribution
    category_counts = df_criterion_category['Category_Group'].value_counts()
    category_dist = (category_counts / criterion_count * 100).to_dict()
    
    return {
        'criterion_count': criterion_count,
        'avg_words': avg_words,
        'avg_attributes': avg_attributes,
        'avg_questions': avg_questions,
        'avg_options': avg_options,
        'category_dist': category_dist
    }


def generate_table1(results: Dict[str, Dict]) -> pd.DataFrame:
    """Generate Table 1: Descriptive Statistics."""
    
    rows = []
    for disease in DISEASES:
        if disease not in results or results[disease] is None:
            continue
        
        r = results[disease]
        rows.append({
            'Disease': disease,
            'Criterion_Count': r['criterion_count'],
            'Words': round(r['avg_words'], 1),
            'Attributes': round(r['avg_attributes'], 1),
            'Questions': round(r['avg_questions'], 1),
            'Options': round(r['avg_options'], 1)
        })
    
    df = pd.DataFrame(rows)
    return df


def generate_table2(results: Dict[str, Dict]) -> pd.DataFrame:
    """Generate Table 2: Criterion Category Distribution (%)."""
    
    # Get all category groups that appear in any disease
    all_categories = set()
    for disease_result in results.values():
        if disease_result is not None:
            all_categories.update(disease_result['category_dist'].keys())
    
    # Remove 'Other' if it exists and add it at the end
    all_categories.discard('Other')
    category_columns = sorted(list(all_categories))
    if 'Other' in [r['category_dist'].keys() for r in results.values() if r]:
        category_columns.append('Other')
    
    rows = []
    for disease in DISEASES:
        if disease not in results or results[disease] is None:
            continue
        
        row = {'Disease': disease}
        dist = results[disease]['category_dist']
        
        for cat in category_columns:
            pct = dist.get(cat, 0.0)
            row[cat] = f"{pct:.1f}%"
        
        rows.append(row)
    
    df = pd.DataFrame(rows)
    return df


# ================================================================
# MAIN EXECUTION
# ================================================================

def main():
    """Generate analysis tables from pipeline outputs."""
    
    print("="*80)
    print("CLINICAL TRIAL CRITERIA ANALYSIS")
    print("="*80)
    print(f"Model: {MODEL}")
    print(f"Diseases: {', '.join(DISEASES)}")
    print("="*80)
    print()
    
    # Ensure output directory exists
    ensure_dir(os.path.join(ROOT_DIR, RESULTS_DIR))
    
    # Analyze each disease
    results = {}
    for disease in DISEASES:
        print(f"Analyzing {disease}...", end=" ")
        result = analyze_disease(disease, MODEL)
        if result:
            results[disease] = result
            print(f"✓ ({result['criterion_count']} criteria)")
        else:
            print("✗ (file not found)")
    
    print()
    
    if not results:
        print("No results to analyze. Check that output files exist.")
        return
    
    # Generate tables
    print("Generating tables...")
    
    # Table 1: Descriptive Statistics
    df_table1 = generate_table1(results)
    table1_path = os.path.join(ROOT_DIR, RESULTS_DIR, f"table1_descriptive_stats_{MODEL}.csv")
    df_table1.to_csv(table1_path, index=False)
    
    print("\n" + "="*80)
    print("TABLE 1: DESCRIPTIVE STATISTICS")
    print("="*80)
    print(df_table1.to_string(index=False))
    print(f"\nSaved to: {table1_path}")
    
    # Table 2: Category Distribution
    df_table2 = generate_table2(results)
    table2_path = os.path.join(ROOT_DIR, RESULTS_DIR, f"table2_category_distribution_{MODEL}.csv")
    df_table2.to_csv(table2_path, index=False)
    
    print("\n" + "="*80)
    print("TABLE 2: CRITERION CATEGORY DISTRIBUTION (%)")
    print("="*80)
    print(df_table2.to_string(index=False))
    print(f"\nSaved to: {table2_path}")
    
    # Save combined Excel file
    excel_path = os.path.join(ROOT_DIR, RESULTS_DIR, f"analysis_results_{MODEL}.xlsx")
    with pd.ExcelWriter(excel_path, engine='openpyxl') as writer:
        df_table1.to_excel(writer, sheet_name='Descriptive_Statistics', index=False)
        df_table2.to_excel(writer, sheet_name='Category_Distribution', index=False)
    
    print(f"\nCombined results saved to: {excel_path}")
    print("\n" + "="*80)
    print("ANALYSIS COMPLETE")
    print("="*80)


if __name__ == "__main__":
    main()

CLINICAL TRIAL CRITERIA ANALYSIS
Model: gpt-5.1
Diseases: Alzheimer, Diabetes, NSCLC, UC, AML, CF, MS

Analyzing Alzheimer... 

/var/folders/w7/7xz416ws3txgypw50xlt1yg00000gn/T/ipykernel_43615/618431143.py:179: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  options_per_criterion = df_valid_options.groupby('Criterion ID').apply(


✓ (114 criteria)
Analyzing Diabetes... 

/var/folders/w7/7xz416ws3txgypw50xlt1yg00000gn/T/ipykernel_43615/618431143.py:179: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  options_per_criterion = df_valid_options.groupby('Criterion ID').apply(


✓ (244 criteria)
Analyzing NSCLC... 

/var/folders/w7/7xz416ws3txgypw50xlt1yg00000gn/T/ipykernel_43615/618431143.py:179: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  options_per_criterion = df_valid_options.groupby('Criterion ID').apply(
/var/folders/w7/7xz416ws3txgypw50xlt1yg00000gn/T/ipykernel_43615/618431143.py:179: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  options_per_criterion = df_valid_options.groupby('Criterion ID').apply(


✓ (118 criteria)
Analyzing UC... ✓ (99 criteria)
Analyzing AML... 

/var/folders/w7/7xz416ws3txgypw50xlt1yg00000gn/T/ipykernel_43615/618431143.py:179: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  options_per_criterion = df_valid_options.groupby('Criterion ID').apply(
/var/folders/w7/7xz416ws3txgypw50xlt1yg00000gn/T/ipykernel_43615/618431143.py:179: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  options_per_criterion = df_valid_options.groupby('Criterion ID').apply(


✓ (116 criteria)
Analyzing CF... ✓ (83 criteria)
Analyzing MS... ✓ (121 criteria)

Generating tables...

TABLE 1: DESCRIPTIVE STATISTICS
  Disease  Criterion_Count  Words  Attributes  Questions  Options
Alzheimer              114   15.1         2.7        1.8      5.2
 Diabetes              244   18.0         2.9        1.9      5.4
    NSCLC              118   16.7         2.6        1.9      6.0
       UC               99   10.7         2.4        1.5      3.7
      AML              116   18.7         2.8        2.4      7.6
       CF               83   16.4         2.2        1.7      4.4
       MS              121   15.1         2.4        1.9      5.0

Saved to: ../output/analysis/table1_descriptive_stats_gpt-5.1.csv

TABLE 2: CRITERION CATEGORY DISTRIBUTION (%)
  Disease Comorbidity Diagnosis Laboratory Neurological Participation Treatment
Alzheimer       34.2%     14.9%       8.8%         5.3%         16.7%     14.9%
 Diabetes       24.2%     13.1%       9.4%         2.5%       

/var/folders/w7/7xz416ws3txgypw50xlt1yg00000gn/T/ipykernel_43615/618431143.py:179: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  options_per_criterion = df_valid_options.groupby('Criterion ID').apply(
